# Privacy for Some — v2 experiments (Colab runner, reviewer revision)

What v2 adds (each item answers a reviewer point):
1. **Name control pools**: rare Anglo names, invented names, Nigerian names, Chinese and Indian names, plus the original common Anglo and Rwandan names (Rev. sSLm Con 1).
2. **Record locale as an explicit factor**: in v1, *every* record had a `+250` phone and a Kigali street, including Western-named ones. v2 crosses each name pool with a Rwandan record and a US record (Rev. sSLm Con 2, Rev. ec5f Weakness 3).
3. **More models**: Qwen2.5 at 1.5B/3B/7B (scale within one family), Phi-3.5-mini, and Llama-3.2-3B (Con 3).
4. **Kinyarwanda comprehension check** (Study C) and an optional Kinyarwanda-capable API model (Con 4).
5. **3 paraphrases per attack**, **format-robust leak matching**, **persona-level statistics**, **per-name breakdown**, **sampling robustness** (minor points).

Runtime: free **T4** is enough for ≤4B models. **Qwen2.5-7B needs an L4 or A100** runtime (fp16 ≈ 15 GB).

## 0. GPU check

In [ ]:
!nvidia-smi -L
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Hugging Face token (needed for gated models such as Llama-3.2) — add `HF_TOKEN` in the 🔑 Secrets panel

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token set')
except Exception:
    print('No HF token — gated models will fail')

## 2. Upload and unpack the project zip (`privacy-exp.zip`)

In [ ]:
from google.colab import files
up = files.upload()
import zipfile, os
z = next(k for k in up if k.endswith('.zip'))
zipfile.ZipFile(z).extractall('/content')
print(os.listdir('/content/privacy-exp'))
%cd /content/privacy-exp

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 4. Check the Kinyarwanda prompts and the name lists (do not skip)
Native speakers: confirm every Kinyarwanda string. **Paraphrases 1–2 (rw) are AI-drafted.** Check them against `RW_VERIFY.md` and fix any wording in `src/prompts.py` (left panel → Files → double-click to edit, Ctrl+S to save), then re-run this cell.
Everyone: confirm the *invented* names in `src/personas.py` are not real names in any language you know.

In [ ]:
from src.prompts import SYSTEM, SETUP, ATTACKS
from src.personas import NAME_POOLS, make_personas
print('SYSTEM (rw):', SYSTEM['rw'], '
')
for label, field, t in ATTACKS:
    print(f'[{label} -> {field}]'); [print('  en:', x) for x in t['en']]; [print('  rw:', x) for x in t['rw']]
for k, v in NAME_POOLS.items():
    print(f"
{k:9s} {v['label']}: ", ', '.join(p['name'] for p in make_personas(30, k)))

## 5. Pipeline test (fake model, ~10 s)

In [ ]:
!python -m src.run_experiment --mock --study A B C --langs en rw --suffix test
!python -m src.analyze results/trialsA_MOCK_test.jsonl
!rm -f results/*MOCK*

## 6. Sanity check: reproduce the v1 numbers (Qwen2.5-3B, original grid, batch size 1, ~600 trials)
This should give 47/150 leaks for the Rwandan name in English vs 2/150 for the Anglo name (strict metric).

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-3B-Instruct --study A --pools en rw --locales rw --langs en rw --paraphrases 1 --batch_size 1 --suffix v1repro
!python -m src.analyze results/trialsA_Qwen25-3B-Instruct_v1repro.jsonl --metric strict
print(open('results/summary.md').read()[:2500])

## 7. MAIN v2 run (English): all 7 name pools × 2 record locales × 5 attacks × 3 paraphrases × 30 personas = 6,300 trials per model
About 1 hour per 3B model on a T4. Run one model per cell so a crash only loses that model.

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-3B-Instruct --study A --suffix main

In [ ]:
!python -m src.run_experiment --model microsoft/Phi-3.5-mini-instruct --study A --suffix main

In [ ]:
!python -m src.run_experiment --model meta-llama/Llama-3.2-3B-Instruct --study A --suffix main   # gated: needs HF token + license accepted

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-1.5B-Instruct --study A --suffix main

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-7B-Instruct --study A --suffix main --batch_size 8   # needs L4/A100

## 8. Kinyarwanda arm, with comprehension check (Study C)
(a) On the small models: this documents the incapacity with evidence instead of an assertion.
(b) Optional, but it answers Reviewer sSLm Con 4: a model that **does** support Kinyarwanda, through an OpenAI-compatible API.
Set `OPENAI_API_KEY` (and `OPENAI_BASE_URL` for OpenRouter/Together, etc.).

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-3B-Instruct --study A C --pools en rw --locales rw us --langs rw --suffix kin
!python -m src.run_experiment --model microsoft/Phi-3.5-mini-instruct --study A C --pools en rw --locales rw us --langs rw --suffix kin

In [ ]:
# Optional API model that handles Kinyarwanda (example; pick one your team has access to)
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# !python -m src.run_experiment --backend openai --model gpt-4o --study A C --pools en rw --locales rw us --langs en rw --suffix kin

## 9. Robustness: sampling instead of greedy (Qwen2.5-3B, T = 0.7, 3 samples, Anglo vs Rwandan only)
Shows the gap is not an artifact of one deterministic decoding path.

In [ ]:
!python -m src.run_experiment --model Qwen/Qwen2.5-3B-Instruct --study A --pools en rw --temperature 0.7 --samples 3 --suffix sampled

## 10. Analyse everything (format-robust metric by default; add `--metric strict` to compare)

In [ ]:
!python -m src.analyze results/trialsA_*_main.jsonl
print(open('results/summary.md').read())
from IPython.display import Image, display; display(Image('results/figure_leak_v2.png'))

In [ ]:
# Kinyarwanda + sampling analyses, written separately
!cp results/summary.md results/summary_main.md
!python -m src.analyze results/trialsA_*_kin.jsonl && cp results/summary.md results/summary_kin.md
!python -m src.analyze results/trialsA_*_sampled.jsonl && cp results/summary.md results/summary_sampled.md
!cp results/summary_main.md results/summary.md

## 11. Native-speaker coding of comprehension (required before reporting Kinyarwanda)
Open each `results/comprehension_coding_*.csv` and fill `human_understood_0_1` (1 = the English translation is correct). Then re-run the Kinyarwanda analysis above; it will use your coding instead of the keyword heuristic.

## 12. Download results

In [ ]:
!zip -qr results_v2.zip results
from google.colab import files; files.download('results_v2.zip')